# Project 04: Prompt Engineering with Google Gemini (and a Look at PEFT)

Write the same task three ways (zero-shot, few-shot, and chain-of-thought), compare the results, and build a prompt from its parts to see how a guardrail changes the output.

Run top to bottom. Each step matches the README.

In [ ]:
import os, subprocess
REPO = "TharunPatel20/GenAI-Engineer"   # the course repo (owner/name)
PROJECT = "projects/04-prompting-and-peft"
FOLDER = REPO.split("/")[1]
if not os.path.isdir(FOLDER):
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"], check=True)
os.chdir(os.path.join(FOLDER, PROJECT))
print("Working in:", os.getcwd())

## Step 1: install the pinned packages

This cell installs the exact versions from `requirements.txt`. Colab may ask you to restart the session. If it does, click **Restart session**, then run the cells again from the top (Step 0 onward).

In [ ]:
%pip install -q google-genai==2.34.0 python-dotenv==1.2.4

## Step 2: API key (Colab Secrets)

Here is how to add your key, in plain words:

1. Click the **key icon** in the left sidebar (Secrets).
2. Click **Add new secret**. Name it exactly `GEMINI_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for that secret.

Never paste the key into a cell. No key yet? Skip this step. The offline demo still runs.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. The offline demo still runs. Details:", e)

print("key found:", bool(os.environ.get("GEMINI_API_KEY", "").strip()))

### Step 1: Load the key safely
**What:** Read `GEMINI_API_KEY` from `.env`, not from the code.
**Why:** Keys written into code get copied, shared, and leaked. A `.env` file stays on your computer.

In Colab the key comes from Colab Secrets (Step 2 above), so this cell reads it from the environment. If the key is missing, the offline demo runs instead of stopping.

In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from google import genai

# Colab has no __file__, so this reads a .env file in the current folder if one exists.
load_dotenv(Path.cwd() / ".env")

# Model names and free-tier limits change. Check Google AI Studio for the current name.
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
api_key = os.getenv("GEMINI_API_KEY", "").strip()
if api_key == "your-key-here":
    api_key = ""

### Step 2: Send one prompt
**What:** Create a client and call the model.
**Why:** Every strategy in this project uses this same call. Only the prompt text changes.

In [ ]:
client = genai.Client(api_key=api_key) if api_key else None

def gemini_reply(client, prompt):
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text

if client:
    print(gemini_reply(client, "Say hello in one word."))
else:
    print("offline demo: no key, so no call was made.")

### Step 3: Zero-shot prompt
**What:** One instruction, no examples.
**Why:** Simple, clear tasks often work with just an instruction. Gemini already understands simple text.

In [ ]:
TASK = "Classify the customer feedback as positive, negative, or neutral."

def zero_shot_prompt(feedback):
    return f"{TASK}\nAnswer with one word only.\n\nFeedback: {feedback}\nLabel:"

### Step 4: Few-shot prompt
**What:** The same instruction, plus two labeled examples before the real feedback.
**Why:** Words such as "neutral" are subjective. Examples show the model what each label means in your case.

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("The app crashes every time I open the cart.", "negative"),
    ("Support solved my issue in five minutes. Thank you!", "positive"),
]

def few_shot_prompt(feedback):
    lines = [TASK, "Answer with one word only.", ""]
    for example_text, example_label in FEW_SHOT_EXAMPLES:
        lines.append(f"Feedback: {example_text}")
        lines.append(f"Label: {example_label}")
        lines.append("")
    lines.append(f"Feedback: {feedback}")
    lines.append("Label:")
    return "\n".join(lines)

### Step 5: Chain-of-thought prompt
**What:** Ask the model to list good points, then bad points, then decide, and to end with a `Final label:` line.
**Why:** Multi-step decisions go more reliably when the model reasons step by step. The cost is a longer answer, which uses more tokens.

In [ ]:
def chain_of_thought_prompt(feedback):
    return (
        f"{TASK}\n"
        "Think step by step: list the good points, then the bad points, "
        "then decide which side is stronger.\n"
        "End with a final line in this form: Final label: <positive|negative|neutral>\n\n"
        f"Feedback: {feedback}"
    )

### Step 6: Read the label
**What:** Find the label word in the reply. If there are several, use the last one.
**Why:** The model may explain itself before the answer. The last label is usually the final answer.

In [ ]:
def extract_label(text):
    matches = re.findall(r"\b(positive|negative|neutral)\b", text, flags=re.IGNORECASE)
    return matches[-1].lower() if matches else "unclear"

### Step 7: Compare side by side
**What:** Run all three prompts on each of three sample feedback lines and print a table with the expected label.
**Why:** Putting the answers next to the expected label is how you judge a prompt. Do not trust one answer.

If no key is set, this table uses canned answers labeled **offline demo**. Those are not real Gemini output.

In [ ]:
FEEDBACK = [
    ("The delivery was fast and the driver was friendly. Great service!", "positive"),
    ("I waited three hours and nobody answered the phone. Very disappointed.", "negative"),
    ("The package arrived on Tuesday, as the tracking page said.", "neutral"),
]

# Offline demo: canned replies. They are NOT real model outputs.
OFFLINE_NOTE = "OFFLINE DEMO: canned replies, not real Gemini output"

def offline_reply(prompt, feedback):
    label = dict(FEEDBACK).get(feedback, "neutral")
    if "step by step" in prompt:
        return f"Good points: fast, friendly. Bad points: none.\nFinal label: {label}"
    return label

if client:
    print(f"Model: {MODEL_NAME}")
else:
    print(f"Mode: {OFFLINE_NOTE}")
print()

header = f"{'Feedback':<38} {'Expected':<9} {'Zero-shot':<10} {'Few-shot':<10} {'CoT':<10}"
print(header)
print("-" * len(header))

for feedback, expected in FEEDBACK:
    labels = []
    for build_prompt in (zero_shot_prompt, few_shot_prompt, chain_of_thought_prompt):
        prompt = build_prompt(feedback)
        if client:
            try:
                text = gemini_reply(client, prompt)
            except Exception as error:  # keep going if one call fails
                print(f"  Call failed: {error}")
                text = ""
        else:
            text = offline_reply(prompt, feedback)
        labels.append(extract_label(text))

    short = feedback if len(feedback) <= 35 else feedback[:32] + "..."
    print(f"{short:<38} {expected:<9} {labels[0]:<10} {labels[1]:<10} {labels[2]:<10}")

print()
print("CoT = chain-of-thought. Compare the columns with the Expected column.")
if not client:
    print(OFFLINE_NOTE + ". Nothing here shows how Gemini would answer.")

### Step 8: Build a prompt from parts
**What:** `build_prompt()` joins persona, instruction, context, tone, guardrail, and output format into one string. The guardrail is optional.
**Why:** Each part can be changed on its own, so prompts are easier to reuse and test.

In [ ]:
PERSONA = "Senior Customer Care Executive"
INSTRUCTION = "Write an email to the customer."
CONTEXT = "The customer was charged twice for the subscription in one month."
TONE = "Empathetic"
GUARDRAIL = "Do not promise a refund. Say that an investigation will take place."
OUTPUT_FORMAT = "A greeting, a body, and a conclusion."

def build_prompt(persona, instruction, context, tone, output_format, guardrail=None):
    lines = [
        f"You are a {persona}.",
        f"Task: {instruction}",
        f"Context: {context}",
        f"Tone: {tone}",
    ]
    if guardrail:
        lines.append(f"Rules you must follow: {guardrail}")
    lines.append(f"Output format: {output_format}")
    return "\n".join(lines)

prompt_plain = build_prompt(PERSONA, INSTRUCTION, CONTEXT, TONE, OUTPUT_FORMAT)
prompt_guarded = build_prompt(PERSONA, INSTRUCTION, CONTEXT, TONE, OUTPUT_FORMAT, GUARDRAIL)

print("PROMPT WITHOUT GUARDRAIL")
print("-" * 40)
print(prompt_plain)
print()
print("PROMPT WITH GUARDRAIL")
print("-" * 40)
print(prompt_guarded)

### Step 9: See what a guardrail changes
**What:** Send the customer-email prompt with and without the guardrail "Do not promise a refund."
**Why:** Guardrails tell the model what it must not do. They are often skipped, and without one the model may promise things you cannot deliver.

If no key is set, the two replies are canned text labeled **offline demo**, not real model output.

In [ ]:
# Offline demo: canned replies. They are NOT real model output.
OFFLINE_WITHOUT_GUARDRAIL = (
    "Dear customer, we are sorry for the double charge. "
    "We have issued a full refund to your card today."
)
OFFLINE_WITH_GUARDRAIL = (
    "Dear customer, we are sorry for the double charge. "
    "We have opened an investigation and will update you within three business days. "
    "We cannot promise a refund until it is complete."
)

if client:
    print(f"Calling Gemini model: {MODEL_NAME}")
    try:
        reply_plain = gemini_reply(client, prompt_plain)
        reply_guarded = gemini_reply(client, prompt_guarded)
    except Exception as error:
        print(f"Call failed: {error}")
        reply_plain, reply_guarded = "", ""
else:
    print("No live call made (no key). Offline demo replies below are canned text, NOT real model output.")
    reply_plain, reply_guarded = OFFLINE_WITHOUT_GUARDRAIL, OFFLINE_WITH_GUARDRAIL

print()
print("REPLY WITHOUT GUARDRAIL")
print("-" * 40)
print(reply_plain)
print()
print("REPLY WITH GUARDRAIL")
print("-" * 40)
print(reply_guarded)
print()
print("Check: does the guarded reply avoid promising a refund?")

## Checkpoints and what you learned

- After Step 2 (API key): `key found: True` if your secret is set, or `key found: False` for the offline demo. No error
- After Step 7 without a key: a table with three rows, labeled `OFFLINE DEMO` at the top and the bottom. No error
- After Step 7 with a key: a table of labels for each feedback line. Compare them with the Expected column
- After Step 9 without a key: two prompts, then two canned replies marked as offline demo
- After Step 9 with a key: two real replies from Gemini. The guarded one should avoid promising a refund

You learned how to keep an API key in Colab Secrets and load it, send a prompt to Gemini with the `google-genai` SDK, write zero-shot, few-shot, and chain-of-thought prompts, build a prompt from parts, and test how a guardrail changes the output.

To keep going, try the stretch goals in the project README, including the PEFT and LoRA reading.

Run locally instead? See the README setup section.